# 4주차. 나나가 기억을 찾아오다

**부제:** Agentic RAG with ChromaDB + SQLite Search Tools

## 0. 목표

이번 주에는 agent가 사용자 질문을 보고 필요한 검색 tool을 고른다. 개인 메모처럼 자연어로 저장된 정보는 ChromaDB RAG 검색 tool로 찾고, 구조화된 일정/할 일/알림 row는 SQLite 검색 tool로 찾는다.

성취 기준:

- RAG 검색 tool과 SQLite 검색 tool의 역할 차이를 말할 수 있다.
- agent가 사용자 질문을 어떤 tool의 `query`로 전달했는지 trace로 확인할 수 있다.
- tool 결과의 `hits` 또는 `rows`가 최종 답변의 근거로 쓰였는지 설명할 수 있다.


![rag](imgs/rag.jpg)

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 프록시 서버를 통해 모델 API를 호출한다. 프록시 토큰, URL, 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → ChromaDB 저장 실습 → Agentic RAG 확장 예제 순서로 진행한다.


In [5]:
# 흐름: repo 설정, 모델 helper, trace helper를 준비한다.
# 4주차에서는 답변보다 ChromaDB/SQLite 검색 hit와 tool trace가 더 중요한 관찰값이다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    # 현재 폴더부터 부모 폴더로 올라가며 repo 루트를 찾는다.
    # 노트북을 notebook/ 안에서 실행해도, repo 루트에서 실행해도 같은 경로를 쓰기 위한 장치다.
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


# 앞으로 나오는 상대 경로는 모두 이 repo 루트를 기준으로 맞춘다.
REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    # 노트북이 repo 루트의 설정 파일을 찾을 수 있게 경로를 추가한다.
    sys.path.insert(0, str(REPO_ROOT))
# 파일 저장/DB 생성 위치가 흔들리지 않도록 작업 폴더를 repo 루트로 고정한다.
os.chdir(REPO_ROOT)

from dotenv import dotenv_values, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
# 이 노트북에서 쓰는 프록시/모델 설정값은 repo 루트의 .env 파일에서만 읽는다.
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)
ENV_VALUES = dotenv_values(ENV_PATH)


def required_env(name: str) -> str:
    value = (ENV_VALUES.get(name) or "").strip()
    if not value or value == "여기에 api key 입력":
        raise RuntimeError(f"repo 루트 .env 파일에 {name}을 설정한 뒤 다시 실행하세요.")
    return value


PROXY_TOKEN = required_env("PROXY_TOKEN")
PROXY_URL = required_env("PROXY_URL")
EMBEDDING_PROXY_URL = required_env("EMBEDDING_PROXY_URL")
OPENAI_MODEL = required_env("OPENAI_MODEL")
OPENAI_EMBEDDING_MODEL = required_env("OPENAI_EMBEDDING_MODEL")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    # temperature=0은 같은 입력에서 비슷한 tool 선택/구조화 결과가 나오게 하기 위한 설정이다.
    return ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=PROXY_TOKEN,
        base_url=PROXY_URL,
        temperature=0,
        max_completion_tokens=max_tokens,
    )


def show_json(value: Any) -> None:
    # ensure_ascii=False로 한글 payload를 사람이 읽기 쉬운 그대로 출력한다.
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    # agent 실행 결과에서 마지막 message가 사용자에게 보이는 최종 답변이다.
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    # LangChain message 전체를 그대로 보여주면 복잡하므로, 수업에서 볼 핵심만 뽑는다.
    trace = []
    for message in agent_result.get("messages", []):
        # tool_calls는 모델이 "이 도구를 이 인자로 실행해줘"라고 요청한 기록이다.
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            # type == "tool"인 message는 실제 tool 실행 결과다.
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: Agentic RAG는 모델이 먼저 검색이 필요한지 판단하고, 필요한 검색 tool에 사용자 질문을 전달한 뒤, tool 결과를 근거로 답하는 흐름이다.

반드시 이해할 것:

- ChromaDB는 자연어 메모를 embedding vector로 저장하고 비슷한 문장을 찾는다.
- `search_rag_memory`는 ChromaDB에서 개인 메모와 참고자료를 검색하는 RAG tool이다.
- `search_sqlite_requests`는 SQLite에 저장된 구조화 요청 row를 검색하는 tool이다.
- agent는 답을 바로 지어내지 않고, 질문 성격에 맞는 검색 tool을 먼저 호출한다.
- 답변 문구보다 `tool_call`, `tool_result`, 검색 결과의 `hits` 또는 `rows`를 먼저 확인한다.

막혔을 때 볼 trace: ChromaDB `count()`, `search_rag_memory` tool call/result, `search_sqlite_requests` tool call/result.


## 3. 기본 개념 실습

가장 작은 흐름은 수강생이 직접 입력한 메모를 ChromaDB persistent collection에 저장하는 것이다. 아직 agent를 붙이지 않고 `client`, `collection`, `add`, `count` 상태만 확인한다.


In [6]:
# 흐름: ChromaDB collection을 만들고 학생 메모를 embedding으로 저장한다.
# count와 collection_name을 확인하면 검색할 기억이 준비됐는지 알 수 있다.
# ChromaDB는 문장을 embedding vector로 저장하고 비슷한 문장을 검색하는 로컬 vector DB다.
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# 수업에서는 사용자가 저장한 기억을 아래 두 문장으로 단순화한다.
student_memories = [
    "프로젝트 발표는 2026-04-24 10:00에 민수와 지아가 함께 진행한다.",
    "카나메이트 UI에서는 채팅 답변과 tool trace를 함께 보여준다.",
]

CHROMA_DIR = REPO_ROOT / "tmp" / "week04_chroma"
COLLECTION_NAME = "kanamate_week4_references"
# persist directory를 만들면 노트북 재실행 후에도 DB 파일 위치를 확인할 수 있다.
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# embedding_function은 텍스트를 vector로 바꾸는 역할을 한다.
embedding_function = OpenAIEmbeddingFunction(
    api_key=PROXY_TOKEN,
    api_base=EMBEDDING_PROXY_URL,
    model_name=OPENAI_EMBEDDING_MODEL,
)
client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(anonymized_telemetry=False))
try:
    # 이전 실행 결과가 남으면 검색 결과가 헷갈리므로 같은 이름의 collection을 지운다.
    client.delete_collection(COLLECTION_NAME)
except Exception as exc:
    if "does not exist" not in str(exc).lower():
        raise

# 새 collection은 위 embedding_function을 사용해 문서를 저장/검색한다.
memory_collection = client.create_collection(name=COLLECTION_NAME, embedding_function=embedding_function)
memory_collection.add(
    # ids는 검색 결과에서 원문을 식별하는 고유 이름이다.
    ids=[f"memory-{index + 1}" for index in range(len(student_memories))],
    documents=student_memories,
    metadatas=[{"source": "student_input", "order": index + 1} for index, _ in enumerate(student_memories)],
)

collection_state = {
    "client_type": type(client).__name__,
    "persist_dir": str(CHROMA_DIR),
    "collection_name": memory_collection.name,
    "count": memory_collection.count(),
}
show_json(collection_state)


{
  "client_type": "Client",
  "persist_dir": "/Users/ssojux2/Working/kakao_clone_coding/tmp/week04_chroma",
  "collection_name": "kanamate_week4_references",
  "count": 2
}


## 4. 카나메이트 확장 예제

agent에게 검색 tool 두 개를 등록한다. `search_rag_memory`는 ChromaDB vector DB에서 자연어 메모를 찾고, `search_sqlite_requests`는 SQLite row에서 구조화된 요청을 찾는다. 두 tool 모두 사용자 질문을 `query`로 받아 검색 결과를 JSON으로 돌려준다.


In [7]:
# 흐름: agent가 호출할 검색 tool 두 개를 만든다.
# search_rag_memory는 ChromaDB RAG 검색, search_sqlite_requests는 SQLite row 검색을 담당한다.
import sqlite3


def format_chroma_results(found: dict[str, Any]) -> list[dict[str, Any]]:
    # Chroma query 결과는 질문 개수 기준으로 한 번 감싸져 있어 [0]으로 첫 질문 결과를 꺼낸다.
    ids = found.get("ids", [[]])[0]
    documents = found.get("documents", [[]])[0]
    distances = found.get("distances", [[]])[0]
    metadatas = found.get("metadatas", [[]])[0]
    return [
        {
            "id": ids[index],
            "content": documents[index],
            "distance": distances[index],
            "metadata": metadatas[index] if index < len(metadatas) and metadatas[index] else {},
        }
        for index in range(len(ids))
    ]


@tool("search_rag_memory", description="ChromaDB vector DB에서 개인 메모와 참고자료를 검색한다. 프로젝트 발표, UI 설명, 사용자가 저장한 기억처럼 자연어 메모에서 사실을 찾아야 할 때 사용한다.")
def search_rag_memory(query: str, top_k: int = 2) -> str:
    """Search free-form student memory with ChromaDB."""
    found = memory_collection.query(query_texts=[query], n_results=top_k)
    hits = format_chroma_results(found)
    return json.dumps({"query": query, "hits": hits}, ensure_ascii=False)


SQLITE_DB_PATH = REPO_ROOT / "tmp" / "week04_saved_requests.sqlite3"
saved_request_rows = [
    {"request_id": "req-demo-1", "kind": "group_schedule", "title": "팀 회의", "date": "2026-05-19", "start_time": "15:00", "members": ["민수", "지아"]},
    {"request_id": "req-demo-2", "kind": "todo", "title": "보고서 제출", "date": "2026-05-22", "priority": "high"},
]


def initialize_sqlite_demo_db() -> None:
    SQLITE_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(SQLITE_DB_PATH) as conn:
        conn.execute("DROP TABLE IF EXISTS saved_requests")
        conn.execute("""
            CREATE TABLE saved_requests (
                request_id TEXT PRIMARY KEY,
                kind TEXT NOT NULL,
                title TEXT NOT NULL,
                payload_json TEXT NOT NULL
            )
        """)
        conn.executemany(
            "INSERT INTO saved_requests (request_id, kind, title, payload_json) VALUES (?, ?, ?, ?)",
            [
                (
                    row["request_id"],
                    row["kind"],
                    row["title"],
                    json.dumps(row, ensure_ascii=False),
                )
                for row in saved_request_rows
            ],
        )
        conn.commit()


initialize_sqlite_demo_db()


@tool("search_sqlite_requests", description="SQLite에 저장된 구조화 요청 row를 검색한다. 저장된 요청, SQLite row, group_schedule, todo, reminder, 구조화 일정/할 일/알림을 물을 때 사용한다.")
def search_sqlite_requests(query: str, top_k: int = 3) -> str:
    """Search structured request rows saved in SQLite."""
    pattern = f"%{query.lower()}%"
    with sqlite3.connect(SQLITE_DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT request_id, kind, title, payload_json
            FROM saved_requests
            WHERE lower(kind) LIKE ? OR lower(title) LIKE ? OR lower(payload_json) LIKE ?
            ORDER BY request_id
            LIMIT ?
            """,
            (pattern, pattern, pattern, top_k),
        ).fetchall()
    hits = []
    for row in rows:
        payload = json.loads(row["payload_json"])
        hits.append({"request_id": row["request_id"], "kind": row["kind"], "title": row["title"], "payload": payload})
    return json.dumps({"query": query, "rows": hits}, ensure_ascii=False)


rag_agent = create_agent(
    model=make_model(700),
    tools=[search_rag_memory, search_sqlite_requests],
    system_prompt=(
        "사용자 질문을 보고 필요한 검색 tool 하나를 먼저 호출한다. "
        "프로젝트 발표, UI, 사용자가 저장한 메모, 참고자료, 기억, 자연어로 저장된 사실은 search_rag_memory를 사용한다. "
        "저장된 요청, SQLite row, 구조화 일정/할 일/알림, kind(group_schedule, todo, reminder)를 묻는 질문은 search_sqlite_requests를 사용한다. "
        "tool의 query에는 사용자 질문을 그대로 넣거나 검색에 필요한 핵심 문구를 넣는다. "
        "tool_result의 hits 또는 rows를 근거로 짧게 답한다."
    ),
)

student_question = "프로젝트 발표는 언제야?"
result = rag_agent.invoke({"messages": [{"role": "user", "content": student_question}]})
student_trace = extract_tool_trace(result)

print(final_text(result))
show_json(student_trace)

assert any(event["event"] == "tool_call" and event["tool_name"] == "search_rag_memory" for event in student_trace)
assert "2026-04-24 10:00" in json.dumps(student_trace, ensure_ascii=False)


프로젝트 발표는 2026년 4월 24일 오전 10시에 민수와 지아가 함께 진행합니다.
[
  {
    "event": "tool_call",
    "tool_name": "search_rag_memory",
    "arguments": {
      "query": "프로젝트 발표 일정"
    }
  },
  {
    "event": "tool_result",
    "tool_name": "search_rag_memory",
    "content": "{\"query\": \"프로젝트 발표 일정\", \"hits\": [{\"id\": \"memory-1\", \"content\": \"프로젝트 발표는 2026-04-24 10:00에 민수와 지아가 함께 진행한다.\", \"distance\": 0.3623676896095276, \"metadata\": {\"source\": \"student_input\", \"order\": 1}}, {\"id\": \"memory-2\", \"content\": \"카나메이트 UI에서는 채팅 답변과 tool trace를 함께 보여준다.\", \"distance\": 0.8829888105392456, \"metadata\": {\"source\": \"student_input\", \"order\": 2}}]}"
  }
]


## 5. 확장 예제 실행

두 질문을 agent에게 던져 어떤 검색 tool이 호출되는지 비교한다. 자연어 메모 질문은 `search_rag_memory`, 저장된 구조화 row 질문은 `search_sqlite_requests`가 호출되어야 한다.


In [8]:
# 흐름: 사용자 질문이 agent를 거쳐 알맞은 검색 tool로 전달되는지 확인한다.
rag_question = "카나메이트 UI에서는 무엇을 함께 보여줘?"
sqlite_question = "저장된 group_schedule row 보여줘"

rag_result = rag_agent.invoke({"messages": [{"role": "user", "content": rag_question}]})
sqlite_result = rag_agent.invoke({"messages": [{"role": "user", "content": sqlite_question}]})

rag_trace = extract_tool_trace(rag_result)
sqlite_trace = extract_tool_trace(sqlite_result)

show_json(collection_state)
print(final_text(rag_result))
show_json(rag_trace)
print(final_text(sqlite_result))
show_json(sqlite_trace)

assert memory_collection.count() == len(student_memories)
assert collection_state["count"] == len(student_memories)
assert any(event["event"] == "tool_call" and event["tool_name"] == "search_rag_memory" for event in rag_trace)
assert any(event["event"] == "tool_result" and "hits" in event["content"] for event in rag_trace)
assert "카나메이트 UI" in json.dumps(rag_trace, ensure_ascii=False)
assert any(event["event"] == "tool_call" and event["tool_name"] == "search_sqlite_requests" for event in sqlite_trace)
assert any(event["event"] == "tool_result" and "rows" in event["content"] for event in sqlite_trace)
assert "group_schedule" in json.dumps(sqlite_trace, ensure_ascii=False)
print("4주차 Agentic RAG 검색 tool 예제 실행 통과")


{
  "client_type": "Client",
  "persist_dir": "/Users/ssojux2/Working/kakao_clone_coding/tmp/week04_chroma",
  "collection_name": "kanamate_week4_references",
  "count": 2
}
카나메이트 UI에서는 채팅 답변과 tool trace를 함께 보여줍니다.
[
  {
    "event": "tool_call",
    "tool_name": "search_rag_memory",
    "arguments": {
      "query": "카나메이트 UI 함께 보여주는 내용"
    }
  },
  {
    "event": "tool_result",
    "tool_name": "search_rag_memory",
    "content": "{\"query\": \"카나메이트 UI 함께 보여주는 내용\", \"hits\": [{\"id\": \"memory-2\", \"content\": \"카나메이트 UI에서는 채팅 답변과 tool trace를 함께 보여준다.\", \"distance\": 0.3215687870979309, \"metadata\": {\"order\": 2, \"source\": \"student_input\"}}, {\"id\": \"memory-1\", \"content\": \"프로젝트 발표는 2026-04-24 10:00에 민수와 지아가 함께 진행한다.\", \"distance\": 0.8605084419250488, \"metadata\": {\"source\": \"student_input\", \"order\": 1}}]}"
  }
]
저장된 group_schedule은 "팀 회의"이며, 날짜는 2026년 5월 19일, 시작 시간은 오후 3시입니다. 참석 멤버는 민수와 지아입니다. 더 필요한 정보가 있나요?
[
  {
    "event": "tool_call",
    "tool_name": "

## 6. 회고

개념 확인 질문:

1. Agentic RAG에서 agent는 언제 tool을 호출하는가?
2. `search_rag_memory`와 `search_sqlite_requests`는 각각 어떤 질문에 맞는가?
3. tool call의 `query`에는 사용자의 질문이 어떻게 전달됐는가?
4. `hits` 또는 `rows`와 최종 답변이 어긋나면 무엇을 의심해야 하는가?

작은 응용 과제: 같은 질문을 개인 메모 질문과 SQLite row 질문으로 바꿔 보고, 호출되는 tool과 tool result가 어떻게 달라지는지 비교한다.
